In [ ]:
#upload kaggle.json
from google.colab import files
files.upload()

Saving kaggle.json to kaggle.json


{'kaggle.json': b'{"username":"abhi2op","key":"adba602b7bd555cd31bb4316c6b46cc5"}'}

In [ ]:
#move kaggle.json
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [ ]:
#download dataset

!kaggle datasets download -d sanikamal/horses-or-humans-dataset

Dataset URL: https://www.kaggle.com/datasets/sanikamal/horses-or-humans-dataset
License(s): other
100% 307M/307M [00:08<00:00, 40.2MB/s]



In [ ]:
#unzip dataset

import zipfile
zip_ref=zipfile.ZipFile("horses-or-humans-dataset.zip")
zip_ref.extractall("/content/dataset")
zip_ref.close()

In [ ]:
#import libraries

import tensorflow as tf
import matplotlib.pyplot as plt
import numpy as np
import os
from tensorflow.keras.preprocessing.image import ImageDataGenerator

In [ ]:
#dataset path

train_dir="/content/dataset/train"
validation_dir="/content/dataset/validation"

In [ ]:
train_dir = "/content/dataset/horse-or-human/train"
validation_dir = "/content/dataset/horse-or-human/validation"

train_datagen=ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    zoom_range=0.2,
    horizontal_flip=True
)
val_datagen=ImageDataGenerator(rescale=1./255)
train_generator=train_datagen.flow_from_directory(
    train_dir,
    target_size=(150,150),
    batch_size=32,
    class_mode='binary')
validation_generator=val_datagen.flow_from_directory(
validation_dir,
 target_size=(150,150),
 batch_size=32,
 class_mode='binary')

Found 1027 images belonging to 2 classes.
Found 256 images belonging to 2 classes.


In [ ]:
# Inspect the dataset directory structure
#!ls -R /content/dataset

In [ ]:
#build CNN Model

model=tf.keras.Sequential([
    tf.keras.layers.Conv2D(32,(3,3), activation='relu', input_shape=(150,150,3)),tf.keras.layers.MaxPooling2D(2,2),
    tf.keras.layers.Conv2D(64,(3,3), activation='relu'),tf.keras.layers.MaxPooling2D(2,2),
    tf.keras.layers.Conv2D(128,(3,3), activation='relu'),tf.keras.layers.MaxPooling2D(2,2),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(512, activation='relu'),
    tf.keras.layers.Dense(1, activation='sigmoid')
])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [ ]:
#compile Model

model.compile(optimizer='adam',loss='binary_crossentropy',metrics=['accuracy'])

In [ ]:
#train model

history=model.fit(train_generator,epochs=10,validation_data=validation_generator)

Epoch 1/10
33/33 ━━━━━━━━━━━━━━━━━━━━ 20s 412ms/step - accuracy: 0.6943 - loss: 0.6837 - val_accuracy: 0.6016 - val_loss: 1.3855
Epoch 2/10
33/33 ━━━━━━━━━━━━━━━━━━━━ 11s 335ms/step - accuracy: 0.9202 - loss: 0.2004 - val_accuracy: 0.7188 - val_loss: 1.3858
Epoch 3/10
33/33 ━━━━━━━━━━━━━━━━━━━━ 11s 335ms/step - accuracy: 0.9406 - loss: 0.1456 - val_accuracy: 0.6719 - val_loss: 2.3874
Epoch 4/10
33/33 ━━━━━━━━━━━━━━━━━━━━ 11s 332ms/step - accuracy: 0.9786 - loss: 0.0780 - val_accuracy: 0.7383 - val_loss: 2.1980
Epoch 5/10
33/33 ━━━━━━━━━━━━━━━━━━━━ 11s 331ms/step - accuracy: 0.9747 - loss: 0.0695 - val_accuracy: 0.7578 - val_loss: 1.8753
Epoch 6/10
33/33 ━━━━━━━━━━━━━━━━━━━━ 20s 341ms/step - accuracy: 0.9766 - loss: 0.0664 - val_accuracy: 0.7930 - val_loss: 1.6166
Epoch 7/10
33/33 ━━━━━━━━━━━━━━━━━━━━ 11s 330ms/step - accuracy: 0.9718 - loss: 0.0710 - val_accuracy: 0.7227 - val_loss: 2.5110
Epoch 8/10
33/33 ━━━━━━━━━━━━━━━━━━━━ 11s 333ms/step - accuracy: 0.9903 - loss: 0.0329 - val_accu

In [ ]:
#evaluate
loss,acc=model.evaluate(validation_generator)
print("Accuracy",acc)
print("Loss",loss)

8/8 ━━━━━━━━━━━━━━━━━━━━ 1s 82ms/step - accuracy: 0.7109 - loss: 2.7990
Accuracy 0.7109375
Loss 2.7989695072174072


In [ ]:
#save model
model.save("horse_human_model.keras")

In [ ]:
#convert to tensorflow lite
converter=tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model=converter.convert()

with open("horse_human_model.tflite","wb") as f:
  f.write(tflite_model)

Saved artifact at '/tmp/tmpfvbsebtu'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 150, 150, 3), dtype=tf.float32, name='keras_tensor_20')
Output Type:
  TensorSpec(shape=(None, 1), dtype=tf.float32, name=None)
Captures:
  132017759941072: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132017759942992: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132017759944528: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132017759941648: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132017759942032: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132017756897744: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132017756898896: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132017756898320: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132017756899280: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132017756899088: TensorSpec(shape=(), dtype=tf.resource, name=None)


In [ ]:
#download TFLite Model
from google.colab import files
files.download("horse_human_model.tflite")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>